# PHƯƠNG ÁN A — ĐÁNH GIÁ LEVEL 1 (BASELINE - KHÔNG DÙNG KHO TRI THỨC) TRÊN FLICKR30K (3 SEEDS: 42, 44, 46)
- **Vai trò trong Phương Án A**: Làm mốc cơ sở (Baseline B0) thuần thị giác trên bộ dữ liệu **Flickr30k Karpathy Test (1,000 ảnh)** qua 3 seeds `[42, 44, 46]` với `Beam Search k=3, alpha=0.6`.
- **Cơ chế hoạt động**: Tự động nạp kết quả/checkpoint Level 1 đã huấn luyện trên Flickr30k (`Level1_Baseline` / `lv1_continue`), chấm đầy đủ **10 chỉ số** (`BLEU-1..4, METEOR, ROUGE-L, CIDEr, SPICE, CHAIR_s, CHAIR_i`) và lưu đồng bộ vào thư mục `knowledge_base_level4_Flick30K` để đối sánh trực tiếp với **Level 2, Level 3, Level 4, Level 5**.

In [1]:
# ==============================================================================
# CELL 1 — CÀI ĐẶT THƯ VIỆN, THIẾT LẬP JAVA 11 & TÀI NGUYÊN SPICE (CORENLP)
# ==============================================================================
import os, sys, subprocess, shutil, urllib.request
from pathlib import Path

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["JAVA_TOOL_OPTIONS"] = "-Duser.language=en -Duser.country=US"

# Dùng subprocess giống PA_B_03 để cài êm, ẩn các cảnh báo phụ thuộc của gradio/diffusers trên Colab
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.1", "faiss-cpu==1.12.0", "pycocoevalcap==1.2",
    "rouge-score", "nltk==3.9.1", "tqdm", "ftfy", "regex", "scipy", "gdown==5.2.2"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

java11_dir = Path("/usr/lib/jvm/java-11-openjdk-amd64")
if not (java11_dir / "bin/java").is_file():
    subprocess.run(["apt-get", "update", "-qq"], check=True, stdout=subprocess.DEVNULL)
    subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-11-jdk-headless"], check=True, stdout=subprocess.DEVNULL)

os.environ["JAVA_HOME"] = str(java11_dir)
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:{os.environ['PATH']}"

if not (java11_dir / "bin/java").is_file():
    raise RuntimeError("Java 11 chưa cài được; kiểm tra lại kết nối mạng Colab.")

# Chuẩn bị sẵn CoreNLP 3.6.0 qua HTTPS cho SPICE (tránh lỗi URL HTTP cũ của pycocoevalcap)
from pycocoevalcap.spice import spice as spice_module
spice_lib = Path(spice_module.__file__).parent / "lib"
spice_lib.mkdir(exist_ok=True)
for jar_name in ["stanford-corenlp-3.6.0.jar", "stanford-corenlp-3.6.0-models.jar"]:
    jar_path = spice_lib / jar_name
    if not jar_path.is_file() or jar_path.stat().st_size < 1_000_000:
        url = "https://repo.maven.apache.org/maven2/edu/stanford/nlp/stanford-corenlp/3.6.0/" + jar_name
        tmp_jar = jar_path.with_name(jar_name + ".tmp")
        print(f"⏳ Đang tải tài nguyên SPICE: {jar_name}...")
        with urllib.request.urlopen(url, timeout=90) as resp, open(tmp_jar, "wb") as out_f:
            shutil.copyfileobj(resp, out_f, length=4 * 1024 * 1024)
        os.replace(tmp_jar, jar_path)

import torch, transformers, faiss, pycocoevalcap
print("✅ Đã thiết lập xong môi trường Python, Java 11 & SPICE CoreNLP!")
print("Torch:", torch.__version__, "| transformers:", transformers.__version__, "| FAISS:", faiss.__version__)

⏳ Đang tải tài nguyên SPICE: stanford-corenlp-3.6.0.jar...
⏳ Đang tải tài nguyên SPICE: stanford-corenlp-3.6.0-models.jar...
✅ Đã thiết lập xong môi trường Python, Java 11 & SPICE CoreNLP!
Torch: 2.11.0+cu130 | transformers: 4.57.1 | FAISS: 1.12.0


In [2]:
# ==============================================================================
# CELL 2 — MOUNT DRIVE, CẤU HÌNH 3 SEEDS [42, 44, 46] & KIỂM TRA SPLIT FLICKR30K
# ==============================================================================
import os, sys, json, time, math, re, random, hashlib, shutil, io
from pathlib import Path
from collections import Counter
from contextlib import redirect_stdout
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

SEEDS = [42, 44, 46]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🌱 Đánh giá Level 1 (Baseline) trên Flickr30k qua 3 SEEDS: {SEEDS} | Thiết bị: {DEVICE}")

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Môi trường:", e)

POSSIBLE_DATA_ROOTS = [
    Path('/content/drive/MyDrive/DataKhoaLuan/Data'),
    Path('/content/drive/MyDrive/Data/Data'),
    Path('/content/drive/MyDrive/Data'),
    Path('/content/drive/MyDrive/KhoaLuanTotNghiep/Data'),
]
DATA_ROOT = next((p for p in POSSIBLE_DATA_ROOTS if (p / 'knowledge_base_level3_Flick30K/index.faiss').exists()), Path('/content/drive/MyDrive/DataKhoaLuan/Data'))
print("✅ DATA_ROOT:", DATA_ROOT)

KB_V2_DIR = DATA_ROOT / 'knowledge_base_level4_Flick30K'
KB_V2_DIR.mkdir(parents=True, exist_ok=True)
PAB_ROOT = DATA_ROOT / 'baseline_level7_Flick30K/PhuongAn_B'

def find_flickr_json(name):
    cands = [
        DATA_ROOT / f'Flickr_Image_dataset-001/flickr30k_images/{name}',
        DATA_ROOT / f'flickr30k_images/{name}',
        DATA_ROOT / name,
    ]
    return next((x for x in cands if x.is_file()), None)

TRAIN_JSON_PATH = find_flickr_json('captions_flickr30k_train.json')
TEST_JSON_PATH  = find_flickr_json('captions_flickr30k_test.json')

with open(TRAIN_JSON_PATH, encoding='utf-8') as f:
    train_data = json.load(f)
with open(TEST_JSON_PATH, encoding='utf-8') as f:
    test_data = json.load(f)

def build_train_vocab():
    counts = Counter()
    for ann in train_data['annotations']:
        counts.update(re.findall(r'\w+|[^\w\s]', ann['caption'].lower()))
    vocab = {'<PAD>': 0, '<BOS>': 1, '<EOS>': 2, '<UNK>': 3}
    for word in sorted(w for w, count in counts.items() if count >= 3):
        if word not in vocab:
            vocab[word] = len(vocab)
    return vocab

stoi = build_train_vocab()
itos = {v: k for k, v in stoi.items()}
BOS_ID, EOS_ID, PAD_ID, UNK_ID = [stoi[k] for k in ['<BOS>', '<EOS>', '<PAD>', '<UNK>']]
print(f"✅ Tập Test: {len(test_data['images']):,} ảnh | Vocab Flickr30k: {len(stoi):,} từ")

🌱 Đánh giá Level 1 (Baseline) trên Flickr30k qua 3 SEEDS: [42, 44, 46] | Thiết bị: cuda
Mounted at /content/drive
✅ DATA_ROOT: /content/drive/MyDrive/DataKhoaLuan/Data
✅ Tập Test: 1,000 ảnh | Vocab Flickr30k: 9,546 từ


In [3]:
# ==============================================================================
# CELL 3 — NẠP ĐẶC TRƯNG TEST, CHẠY/NẠP DỰ ĐOÁN LEVEL 1 QUA 3 SEEDS & CHẤM 10 CHỈ SỐ
# ==============================================================================
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice
# CELL ĐÁNH GIÁ — CHỈ CHẠY SAU KHI TRAIN HOÀN TẤT
# ==============================================================================
# CELL 6 — ĐÁNH GIÁ CHÍNH THỨC TRÊN TẬP TEST BẰNG BEAM SEARCH (10 ĐỘ ĐO COCO + CHAIR)
# ==============================================================================
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice


# Extracted without logic changes from clips/pattern pattern/text/en/inflect.py,
# singularize() and its rule tables (see upstream license and attribution).
# https://github.com/clips/pattern/blob/master/pattern/text/en/inflect.py
import re
NOUN = 'NN'
plural_prepositions = set('about before during of till above behind except off to across below for on under after beneath from onto until among beside in out unto around besides into over upon at between near since with athwart betwixt beyond but by'.split())
#### SINGULARIZE ###################################################################################
# Adapted from Bermi Ferrer's Inflector for Python:
# http://www.bermi.org/inflector/
# Copyright (c) 2006 Bermi Ferrer Martinez
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software to deal in this software without restriction, including
# without limitation the rights to use, copy, modify, merge, publish,
# distribute, sublicense, and/or sell copies of this software, and to permit
# persons to whom this software is furnished to do so, subject to the following
# condition:
#
# THIS SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THIS SOFTWARE OR THE USE OR OTHER DEALINGS IN
# THIS SOFTWARE.
singular_rules = [
    (r'(?i)(.)ae$'            , '\\1a'    ),
    (r'(?i)(.)itis$'          , '\\1itis' ),
    (r'(?i)(.)eaux$'          , '\\1eau'  ),
    (r'(?i)(quiz)zes$'        , '\\1'     ),
    (r'(?i)(matr)ices$'       , '\\1ix'   ),
    (r'(?i)(ap|vert|ind)ices$', '\\1ex'   ),
    (r'(?i)^(ox)en'           , '\\1'     ),
    (r'(?i)(alias|status)es$' , '\\1'     ),
    (r'(?i)([octop|vir])i$'   , '\\1us'  ),
    (r'(?i)(cris|ax|test)es$' , '\\1is'   ),
    (r'(?i)(shoe)s$'          , '\\1'     ),
    (r'(?i)(o)es$'            , '\\1'     ),
    (r'(?i)(bus)es$'          , '\\1'     ),
    (r'(?i)([m|l])ice$'       , '\\1ouse' ),
    (r'(?i)(x|ch|ss|sh)es$'   , '\\1'     ),
    (r'(?i)(m)ovies$'         , '\\1ovie' ),
    (r'(?i)(.)ombies$'        , '\\1ombie'),
    (r'(?i)(s)eries$'         , '\\1eries'),
    (r'(?i)([^aeiouy]|qu)ies$', '\\1y'    ),
        # -f, -fe sometimes take -ves in the plural
        # (e.g., lives, wolves).
    (r"([aeo]l)ves$"          , "\\1f"    ),
    (r"([^d]ea)ves$"          , "\\1f"    ),
    (r"arves$"                , "arf"     ),
    (r"erves$"                , "erve"    ),
    (r"([nlw]i)ves$"          , "\\1fe"   ),
    (r'(?i)([lr])ves$'        , '\\1f'    ),
    (r"([aeo])ves$"           , "\\1ve"   ),
    (r'(?i)(sive)s$'          , '\\1'     ),
    (r'(?i)(tive)s$'          , '\\1'     ),
    (r'(?i)(hive)s$'          , '\\1'     ),
    (r'(?i)([^f])ves$'        , '\\1fe'   ),
    # -ses suffixes.
    (r'(?i)(^analy)ses$'      , '\\1sis'  ),
    (r'(?i)((a)naly|(b)a|(d)iagno|(p)arenthe|(p)rogno|(s)ynop|(t)he)ses$', '\\1\\2sis'),
    (r'(?i)(.)opses$'         , '\\1opsis'),
    (r'(?i)(.)yses$'          , '\\1ysis' ),
    (r'(?i)(h|d|r|o|n|b|cl|p)oses$', '\\1ose'),
    (r'(?i)(fruct|gluc|galact|lact|ket|malt|rib|sacchar|cellul)ose$', '\\1ose'),
    (r'(?i)(.)oses$'          , '\\1osis' ),
    # -a
    (r'(?i)([ti])a$'          , '\\1um'   ),
    (r'(?i)(n)ews$'           , '\\1ews'  ),
    (r'(?i)s$'                , ''        ),
]
# For performance, compile the regular expressions only once:
singular_rules = [(re.compile(r[0]), r[1]) for r in singular_rules]
singular_uninflected = set((
    "bison"      , "debris"   , "headquarters", "pincers"    , "trout"     ,
    "bream"      , "diabetes" , "herpes"      , "pliers"     , "tuna"      ,
    "breeches"   , "djinn"    , "high-jinks"  , "proceedings", "whiting"   ,
    "britches"   , "eland"    , "homework"    , "rabies"     , "wildebeest",
    "carp"       , "elk"      , "innings"     , "salmon"     ,
    "chassis"    , "flounder" , "jackanapes"  , "scissors"   ,
    "christmas"  , "gallows"  , "mackerel"    , "series"     ,
    "clippers"   , "georgia"  , "measles"     , "shears"     ,
    "cod"        , "graffiti" , "mews"        , "species"    ,
    "contretemps",              "mumps"       , "swine"      ,
    "corps"      ,              "news"        , "swiss"      ,
))
singular_uncountable = set((
    "advice"     , "equipment", "happiness"   , "luggage"    , "news"      , "software"     ,
    "bread"      , "fruit"    , "information" , "mathematics", "progress"  , "understanding",
    "butter"     , "furniture", "ketchup"     , "mayonnaise" , "research"  , "water"        ,
    "cheese"     , "garbage"  , "knowledge"   , "meat"       , "rice"      ,
    "electricity", "gravel"   , "love"        , "mustard"    , "sand"      ,
))
singular_ie = set((
    "alergie"    , "cutie"    , "hoagie"      , "newbie"     , "softie"    , "veggie"       ,
    "auntie"     , "doggie"   , "hottie"      , "nightie"    , "sortie"    , "weenie"       ,
    "beanie"     , "eyrie"    , "indie"       , "oldie"      , "stoolie"   , "yuppie"       ,
    "birdie"     , "freebie"  , "junkie"      , "^pie"       , "sweetie"   , "zombie"       ,
    "bogie"      , "goonie"   , "laddie"      , "pixie"      , "techie"    ,
    "bombie"     , "groupie"  , "laramie"     , "quickie"    , "^tie"      ,
    "collie"     , "hankie"   , "lingerie"    , "reverie"    , "toughie"   ,
    "cookie"     , "hippie"   , "meanie"      , "rookie"     , "valkyrie"  ,
))
singular_irregular = {
       "atlantes": "atlas",
        "atlases": "atlas",
           "axes": "axe",
         "beeves": "beef",
       "brethren": "brother",
       "children": "child",
        "corpora": "corpus",
       "corpuses": "corpus",
    "ephemerides": "ephemeris",
           "feet": "foot",
        "ganglia": "ganglion",
          "geese": "goose",
         "genera": "genus",
          "genii": "genie",
       "graffiti": "graffito",
         "helves": "helve",
           "kine": "cow",
         "leaves": "leaf",
         "loaves": "loaf",
            "men": "man",
      "mongooses": "mongoose",
         "monies": "money",
          "moves": "move",
         "mythoi": "mythos",
         "numena": "numen",
       "occipita": "occiput",
      "octopodes": "octopus",
          "opera": "opus",
         "opuses": "opus",
            "our": "my",
           "oxen": "ox",
          "penes": "penis",
        "penises": "penis",
         "people": "person",
          "sexes": "sex",
    "soliloquies": "soliloquy",
          "teeth": "tooth",
         "testes": "testis",
        "trilbys": "trilby",
         "turves": "turf",
            "zoa": "zoon",
}

def singularize(word, pos=NOUN, custom={}):
    """ Returns the singular of a given word.
    """
    if word in custom:
        return custom[word]
    # Recurse compound words (e.g. mothers-in-law).
    if "-" in word:
        w = word.split("-")
        if len(w) > 1 and w[1] in plural_prepositions:
            return singularize(w[0], pos, custom) + "-" + "-".join(w[1:])
    # dogs' => dog's
    if word.endswith("'"):
        return singularize(word[:-1]) + "'s"
    w = word.lower()
    for x in singular_uninflected:
        if x.endswith(w):
            return word
    for x in singular_uncountable:
        if x.endswith(w):
            return word
    for x in singular_ie:
        if w.endswith(x + "s"):
            return w
    for x in singular_irregular:
        if w.endswith(x):
            return re.sub('(?i)' + x + '$', singular_irregular[x], word)
    for suffix, inflection in singular_rules:
        m = suffix.search(word)
        g = m and m.groups() or []
        if m:
            for k in range(len(g)):
                if g[k] is None:
                    inflection = inflection.replace('\\' + str(k + 1), '')
            return suffix.sub(inflection, word)
    return word


SYNONYMS_RAW = "person, girl, boy, man, woman, kid, child, chef, baker, people, adult, rider, children, baby, worker, passenger, sister, biker, policeman, cop, officer, lady, cowboy, bride, groom, male, female, guy, traveler, mother, father, gentleman, pitcher, player, skier, snowboarder, skater, skateboarder, person, woman, guy, foreigner, child, gentleman, caller, offender, coworker, trespasser, patient, politician, soldier, grandchild, serviceman, walker, drinker, doctor, bicyclist, thief, buyer, teenager, student, camper, driver, solider, hunter, shopper, villager\nbicycle, bike, bicycle, bike, unicycle, minibike, trike\ncar, automobile, van, minivan, sedan, suv, hatchback, cab, jeep, coupe, taxicab, limo, taxi\nmotorcycle, scooter,  motor bike, motor cycle, motorbike, scooter, moped\nairplane, jetliner, plane, air plane, monoplane, aircraft, jet, jetliner, airbus, biplane, seaplane\nbus, minibus, trolley\ntrain, locomotive, tramway, caboose\ntruck, pickup, lorry, hauler, firetruck\nboat, ship, liner, sailboat, motorboat, dinghy, powerboat, speedboat, canoe, skiff, yacht, kayak, catamaran, pontoon, houseboat, vessel, rowboat, trawler, ferryboat, watercraft, tugboat, schooner, barge, ferry, sailboard, paddleboat, lifeboat, freighter, steamboat, riverboat, battleship, steamship\ntraffic light, street light, traffic signal, stop light, streetlight, stoplight\nfire hydrant, hydrant\nstop sign\nparking meter\nbench, pew\nbird, ostrich, owl, seagull, goose, duck, parakeet, falcon, robin, pelican, waterfowl, heron, hummingbird, mallard, finch, pigeon, sparrow, seabird, osprey, blackbird, fowl, shorebird, woodpecker, egret, chickadee, quail, bluebird, kingfisher, buzzard, willet, gull, swan, bluejay, flamingo, cormorant, parrot, loon, gosling, waterbird, pheasant, rooster, sandpiper, crow, raven, turkey, oriole, cowbird, warbler, magpie, peacock, cockatiel, lorikeet, puffin, vulture, condor, macaw, peafowl, cockatoo, songbird\ncat, kitten, feline, tabby\ndog, puppy, beagle, pup, chihuahua, schnauzer, dachshund, rottweiler, canine, pitbull, collie, pug, terrier, poodle, labrador, doggie, doberman, mutt, doggy, spaniel, bulldog, sheepdog, weimaraner, corgi, cocker, greyhound, retriever, brindle, hound, whippet, husky\nhorse, colt, pony, racehorse, stallion, equine, mare, foal, palomino, mustang, clydesdale, bronc, bronco\nsheep, lamb, ram, lamb, goat, ewe\ncow, cattle, oxen, ox, calf, cattle, holstein, heifer, buffalo, bull, zebu, bison\nelephant\nbear, panda\nzebra\ngiraffe\nbackpack, knapsack\numbrella\nhandbag, wallet, purse, briefcase\ntie, bow, bow tie\nsuitcase, suit case, luggage\nfrisbee\nskis, ski\nsnowboard\nsports ball, ball\nkite\nbaseball bat\nbaseball glove\nskateboard\nsurfboard, longboard, skimboard, shortboard, wakeboard\ntennis racket, racket\nbottle\nwine glass\ncup\nfork\nknife, pocketknife, knive\nspoon\nbowl, container\nbanana\napple\nsandwich, burger, sub, cheeseburger, hamburger\norange\nbroccoli\ncarrot\nhot dog\npizza\ndonut, doughnut, bagel\ncake,  cheesecake, cupcake, shortcake, coffeecake, pancake\nchair, seat, stool\ncouch, sofa, recliner, futon, loveseat, settee, chesterfield\npotted plant, houseplant\nbed\ndining table, table, desk\ntoilet, urinal, commode, toilet, lavatory, potty\ntv, monitor, televison, television\nlaptop, computer, notebook, netbook, lenovo, macbook, laptop computer\nmouse\nremote\nkeyboard\ncell phone, mobile phone, phone, cellphone, telephone, phon, smartphone, iPhone\nmicrowave\noven, stovetop, stove, stove top oven\ntoaster\nsink\nrefrigerator, fridge, fridge, freezer\nbook\nclock\nvase\nscissors\nteddy bear, teddybear\nhair drier, hairdryer\ntoothbrush\n"

DOUBLE_WORDS = [
    'motor bike', 'motor cycle', 'air plane', 'traffic light', 'street light',
    'traffic signal', 'stop light', 'fire hydrant', 'stop sign', 'parking meter',
    'suit case', 'sports ball', 'baseball bat', 'baseball glove', 'tennis racket',
    'wine glass', 'hot dog', 'cell phone', 'mobile phone', 'teddy bear',
    'hair drier', 'potted plant', 'bow tie', 'laptop computer', 'stove top oven',
    'home plate', 'train track'
]
ANIMALS = ['bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'animal', 'cub']
VEHICLES = ['jet', 'train']

class StandaloneCHAIRScorer:
    def __init__(self, raw_synonyms):
        lines = [l.strip() for l in raw_synonyms.splitlines() if l.strip()]
        synonyms = [l.split(', ') for l in lines]
        self.mscoco_objects = set(w for row in synonyms for w in row)
        self.inverse_synonyms = {w: row[0] for row in synonyms for w in row}
        self.double_words = {w: w for w in DOUBLE_WORDS}
        for a in ANIMALS:
            self.double_words['baby ' + a] = a
            self.double_words['adult ' + a] = a
        for v in VEHICLES:
            self.double_words['passenger ' + v] = v
        self.double_words.update({'bow tie': 'tie', 'toilet seat': 'toilet', 'wine glas': 'wine glass'})

    def caption_to_words(self, caption):
        raw_words = re.findall(r'\b[a-zA-Z]+\b', caption.lower())
        words = [singularize(w) for w in raw_words]
        collapsed = []
        i = 0
        while i < len(words):
            phrase = ' '.join(words[i:i+2]) if i + 1 < len(words) else ''
            if phrase in self.double_words:
                collapsed.append(self.double_words[phrase])
                i += 2
            else:
                collapsed.append(words[i])
                i += 1
        if 'toilet' in collapsed and 'seat' in collapsed:
            collapsed = [w for w in collapsed if w != 'seat']
        return [self.inverse_synonyms[w] for w in collapsed if w in self.mscoco_objects]

    def compute_chair(self, ground_truth_dict, preds_list):
        gt_objects = {}
        for iid, caps in ground_truth_dict.items():
            objs = set()
            for c in caps:
                objs.update(self.caption_to_words(c))
            gt_objects[iid] = objs

        num_sentences = len(preds_list)
        hallucinated_sentences = 0
        total_objects = 0
        hallucinated_objects = 0

        for item in preds_list:
            iid = item['image_id']
            pred_objs = self.caption_to_words(item['caption'])
            gt_set = gt_objects.get(iid, set())
            cur_hal = [obj for obj in pred_objs if obj not in gt_set]

            total_objects += len(pred_objs)
            hallucinated_objects += len(cur_hal)
            if len(cur_hal) > 0:
                hallucinated_sentences += 1

        chair_s = (hallucinated_sentences / num_sentences * 100.0) if num_sentences > 0 else 0.0
        chair_i = (hallucinated_objects / total_objects * 100.0) if total_objects > 0 else 0.0
        return {'CHAIR_s': float(chair_s), 'CHAIR_i': float(chair_i)}

chair_scorer = StandaloneCHAIRScorer(SYNONYMS_RAW)


import subprocess, io
from contextlib import redirect_stdout

class ManagedMeteor(Meteor):
    """Đóng tiến trình Java Meteor an toàn sau mỗi seed để không giữ lock/RAM khi chạy qua nhiều seed."""
    def close(self):
        process = getattr(self, 'meteor_p', None)
        if process is None:
            return
        if process.poll() is None:
            process.kill()
        process.wait(timeout=10)
        for stream in [process.stdin, process.stdout, process.stderr]:
            if stream is not None:
                try:
                    stream.close()
                except OSError:
                    pass
    def __del__(self):
        try:
            self.close()
        except Exception:
            pass

# ==============================================================================
# CƠ CHẾ HIỆU CHỈNH BỘ NHỚ ĐA TOKEN & CỔNG TỰ NHIÊN VỚI BEAM SEARCH CHUẨN (k=3, alpha=0.6)
# Giữ nguyên 100% Fair Benchmark (beam_size=3, alpha=0.6) giống hệt Level 1, Level 2, Level 3
# ==============================================================================

class BaselineCaptioner(nn.Module):
    def __init__(self, clip_dim=512, vocab_size=len(stoi), d_model=256, nhead=8, num_layers=3, dropout=0.1, max_len=30):
        super().__init__()
        self.d_model = d_model
        self.feature_projection = nn.Linear(clip_dim, d_model)
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.position_embedding = nn.Embedding(max_len, d_model)
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=1024, dropout=dropout, batch_first=True)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.output_projection = nn.Linear(d_model, vocab_size)

    def forward_with_memory(self, memory, tokens):
        B, T = tokens.size()
        pos = torch.arange(T, device=tokens.device).unsqueeze(0).expand(B, -1)
        token_emb = self.token_embedding(tokens) + self.position_embedding(pos)
        tgt_mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=tokens.device), diagonal=1)
        pad_mask = tokens.eq(PAD_ID)
        out = self.decoder(token_emb, memory.expand(B, -1, -1), tgt_mask=tgt_mask, tgt_key_padding_mask=pad_mask)
        return self.output_projection(out)

test_meta = test_data['images']
feat_cands = [
    DATA_ROOT / 'cache_clip_flickr30k/test_features.npy',
    DATA_ROOT / 'baseline_level7_Flick30K/PhuongAn_A/flickr30k_test_clip_features.npy',
]
test_feats = None
for fp in feat_cands:
    if fp.is_file():
        arr = np.load(str(fp))
        if arr.shape == (1000, 512):
            norms = np.linalg.norm(arr, axis=-1, keepdims=True)
            test_feats = (arr / np.maximum(norms, 1e-12)).astype(np.float32)
            print(f"✅ Đã nạp đặc trưng CLIP 1,000 ảnh Test từ: {fp}")
            break

gts = {}
gts_raw = {}
for ann in test_data['annotations']:
    gts.setdefault(ann['image_id'], []).append({'caption': ann['caption']})
    gts_raw.setdefault(ann['image_id'], []).append(ann['caption'])

tokenizer = PTBTokenizer()
gts_tok = tokenizer.tokenize(gts)

def token_ids_to_text(ids):
    words = []
    for token in ids:
        i = int(token)
        if i == EOS_ID:
            break
        if i not in {PAD_ID, BOS_ID}:
            words.append(itos[i])
    return ' '.join(words)

def load_lv1_flickr_model(seed_val):
    warm_epochs = PAB_ROOT / f'WarmStart_LV1_LV5/paired_warmstart_20261018_safe_v2/seed_{seed_val}/lv1_continue/epochs'
    cands = []
    if warm_epochs.is_dir():
        cands.extend(sorted(warm_epochs.glob('epoch_*.pt'), reverse=True))
    cands.extend([
        PAB_ROOT / f'Level1_Baseline/v4_checked_20261006/seed_{seed_val}/best_model.pt',
        PAB_ROOT / f'Level1_Baseline/seed_{seed_val}/best_model.pt',
        PAB_ROOT / f'Level1_Baseline/seed_{seed_val}/checkpoint_best.pt',
    ])
    for p in cands:
        if p.is_file():
            try:
                cp = torch.load(p, map_location='cpu', weights_only=False)
                state = cp['best_snapshot']['model_state_dict'] if 'best_snapshot' in cp else cp.get('model_state_dict', cp)
                m = BaselineCaptioner().to(DEVICE)
                m.load_state_dict(state, strict=True)
                m.eval()
                return m, p
            except Exception:
                pass
    return None, None

@torch.inference_mode()
def generate_lv1_beam(model, feat_np, beam_size=3, alpha=0.6, max_len=30):
    v = torch.from_numpy(feat_np).unsqueeze(0).to(DEVICE)
    mem = model.feature_projection(v).unsqueeze(1)
    beams = [([BOS_ID], 0.0, False)]
    def rank(item):
        return item[1] / (((5.0 + max(1, len(item[0]) - 1)) / 6.0) ** alpha)
    for _ in range(max_len - 1):
        act = [idx for idx, b in enumerate(beams) if not b[2]]
        if not act:
            break
        inp = torch.tensor([beams[idx][0] for idx in act], dtype=torch.long, device=DEVICE)
        logits = model.forward_with_memory(mem, inp)
        logp = F.log_softmax(logits[:, -1].float(), dim=-1)
        logp[:, [PAD_ID, BOS_ID]] = -torch.inf
        cands = []
        ap = 0
        for b_idx, (ids, sc, done) in enumerate(beams):
            if done:
                cands.append((ids, sc, True))
                continue
            vals, idxs = logp[ap].topk(beam_size)
            ap += 1
            for val_v, idx_v in zip(vals.tolist(), idxs.tolist()):
                cands.append((ids + [idx_v], sc + val_v, idx_v == EOS_ID))
        beams = sorted(cands, key=rank, reverse=True)[:beam_size]
        if all(b[2] for b in beams):
            break
    best = max(beams, key=rank)
    return token_ids_to_text(best[0][1:])

ALL_SEED_METRICS = {}
for cur_seed in SEEDS:
    print("\n" + "=" * 80)
    print(f"🔍 ĐANG ĐÁNH GIÁ LEVEL 1 (BASELINE) TRÊN FLICKR30K | SEED {cur_seed}")
    print("=" * 80)

    # Kiểm tra nếu đã có file metrics chuẩn của Level 1 trên Drive thì nạp ngay
    existing_metric_cands = [
        PAB_ROOT / f'Level1_Baseline/v4_checked_20261006/seed_{cur_seed}/level1_in_domain_metrics.json',
        PAB_ROOT / f'Level1_Baseline/seed_{cur_seed}/level1_in_domain_metrics.json',
    ]
    warm_dir = PAB_ROOT / f'WarmStart_LV1_LV5/paired_warmstart_20261018_safe_v2/seed_{cur_seed}/lv1_continue'
    if warm_dir.is_dir():
        existing_metric_cands.extend(sorted(warm_dir.glob('test_metrics_*.json'), reverse=True))

    loaded_m = None
    for mp in existing_metric_cands:
        if mp.is_file():
            try:
                doc = json.loads(mp.read_text(encoding='utf-8'))
                m_dict = doc.get('metrics', doc)
                if 'CIDEr' in m_dict and 'CHAIR_s' in m_dict:
                    loaded_m = {k: float(v) for k, v in m_dict.items()}
                    print(f"♻️ [Seed {cur_seed}] Đã nạp kết quả chuẩn Level 1 từ Drive: {mp.name}")
                    break
            except Exception:
                pass

    if loaded_m is None:
        m_lv1, ck_path = load_lv1_flickr_model(cur_seed)
        if m_lv1 is None:
            raise FileNotFoundError(f"Không tìm thấy checkpoint Level 1 cho Seed {cur_seed}")
        preds = []
        for i in tqdm(range(len(test_feats)), desc=f"[Seed {cur_seed}] LV1 Beam Search (k=3)"):
            preds.append({'image_id': test_meta[i]['id'], 'caption': generate_lv1_beam(m_lv1, test_feats[i])})
        res = {p['image_id']: [{'caption': p['caption']}] for p in preds}
        res_tok = tokenizer.tokenize(res)
        loaded_m = {}
        for scorer_cls, method in [
            (Bleu, ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
            (ManagedMeteor, "METEOR"),
            (Rouge, "ROUGE_L"),
            (Cider, "CIDEr"),
            (Spice, "SPICE")
        ]:
            sc_obj = scorer_cls(4) if scorer_cls is Bleu else scorer_cls()
            try:
                with redirect_stdout(io.StringIO()):
                    score, _ = sc_obj.compute_score(gts_tok, res_tok)
                if isinstance(method, list):
                    for m_n, s_v in zip(method, score):
                        loaded_m[m_n] = float(s_v * 100)
                else:
                    loaded_m[method] = float(score * 100)
            finally:
                if isinstance(sc_obj, ManagedMeteor):
                    sc_obj.close()
                del sc_obj
        ch = chair_scorer.compute_chair(gts_raw, preds)
        loaded_m['CHAIR_s'] = ch['CHAIR_s']
        loaded_m['CHAIR_i'] = ch['CHAIR_i']

    ALL_SEED_METRICS[cur_seed] = loaded_m
    for k, v in loaded_m.items():
        print(f"  * {k:12s}: {v:8.2f}")

full_order = ["CIDEr", "Bleu_4", "Bleu_1", "Bleu_2", "Bleu_3", "METEOR", "ROUGE_L", "SPICE", "CHAIR_s", "CHAIR_i"]
print("\n" + "=" * 84)
print(f"🏆 TỔNG HỢP LEVEL 1 (BASELINE - KHÔNG DÙNG KB) TRÊN 3 SEED: {SEEDS}")
print("=" * 84)
print(f"| {'Chỉ số':<10} | {'Seed 42':>9} | {'Seed 44':>9} | {'Seed 46':>9} | {'Mean ± SD':>15} |")
print(f"| {'-'*10} | {'-'*9} | {'-'*9} | {'-'*9} | {'-'*15} |")
summary_lv1 = {}
for k in full_order:
    vals = np.array([ALL_SEED_METRICS[s][k] for s in SEEDS], dtype=np.float64)
    m_v, sd_v = float(vals.mean()), float(vals.std(ddof=1))
    summary_lv1[k] = {'mean': round(m_v, 2), 'sd': round(sd_v, 2)}
    print(f"| {k:<10} | {ALL_SEED_METRICS[42][k]:9.2f} | {ALL_SEED_METRICS[44][k]:9.2f} | {ALL_SEED_METRICS[46][k]:9.2f} | {m_v:6.2f} ± {sd_v:4.2f}   |")
print("=" * 84)

out_lv1_path = KB_V2_DIR / 'ketqua_danhgia_level1_baseline_flickr30k_3seeds.json'
with open(out_lv1_path, 'w', encoding='utf-8') as f:
    json.dump({'level': 1, 'seeds': SEEDS, 'per_seed_metrics': ALL_SEED_METRICS, 'summary_metrics': summary_lv1}, f, ensure_ascii=False, indent=2)
print(f"✅ Đã lưu kết quả tổng hợp Level 1 tại: {out_lv1_path}")

✅ Đã nạp đặc trưng CLIP 1,000 ảnh Test từ: /content/drive/MyDrive/DataKhoaLuan/Data/cache_clip_flickr30k/test_features.npy

🔍 ĐANG ĐÁNH GIÁ LEVEL 1 (BASELINE) TRÊN FLICKR30K | SEED 42
♻️ [Seed 42] Đã nạp kết quả chuẩn Level 1 từ Drive: level1_in_domain_metrics.json
  * Bleu_1      :    69.28
  * Bleu_2      :    50.96
  * Bleu_3      :    36.66
  * Bleu_4      :    26.17
  * METEOR      :    21.25
  * ROUGE_L     :    48.29
  * CIDEr       :    54.27
  * SPICE       :    15.01
  * CHAIR_s     :    13.10
  * CHAIR_i     :     9.46

🔍 ĐANG ĐÁNH GIÁ LEVEL 1 (BASELINE) TRÊN FLICKR30K | SEED 44
♻️ [Seed 44] Đã nạp kết quả chuẩn Level 1 từ Drive: level1_in_domain_metrics.json
  * Bleu_1      :    68.74
  * Bleu_2      :    50.48
  * Bleu_3      :    36.24
  * Bleu_4      :    25.78
  * METEOR      :    21.02
  * ROUGE_L     :    47.98
  * CIDEr       :    53.14
  * SPICE       :    14.81
  * CHAIR_s     :    13.00
  * CHAIR_i     :     9.53

🔍 ĐANG ĐÁNH GIÁ LEVEL 1 (BASELINE) TRÊN FLICKR30K 